# Bash Kernel Test Notebook

This notebook demonstrates the new bash language support in .NET Interactive with:
- Basic bash execution
- Persistent sessions (cd, export persist across cells)
- Bidirectional variable sharing with C# and other kernels

## Use a local build of dotnet-interactive

In VS Code, Polyglot Notebooks launches the .NET Interactive kernel based on user settings. The notebook file cannot directly override the kernel binary path, so you switch to a local build by updating VS Code settings.

1. Build the local tool (repo root):

   ```bash
   dotnet build src/dotnet-interactive/dotnet-interactive.csproj
   ```

2. The built app DLL in this container is:

   ```
   /workspaces/interactive/artifacts/bin/dotnet-interactive/Debug/net10.0/Microsoft.DotNet.Interactive.App.dll
   ```

3. Open `Preferences: Open Settings (JSON)` and add:

   ```json
   "dotnet-interactive.kernelTransportArgs": [
     "{dotnet_path}",
     "/workspaces/interactive/artifacts/bin/dotnet-interactive/Debug/net10.0/Microsoft.DotNet.Interactive.App.dll",
     "[vscode]",
     "stdio"
   ],

   "dotnet-interactive.notebookParserArgs": [
     "{dotnet_path}",
     "/workspaces/interactive/artifacts/bin/dotnet-interactive/Debug/net10.0/Microsoft.DotNet.Interactive.App.dll",
     "notebook-parser"
   ]
   ```

4. Restart VS Code, then restart the notebook kernel.

Run the next cell to confirm you are running your local build.

In [ ]:
#!csharp
using System;
using System.Reflection;
using Microsoft.DotNet.Interactive;

Console.WriteLine($"EntryAssembly: {Assembly.GetEntryAssembly()?.Location}");
Console.WriteLine($"BaseDirectory: {AppContext.BaseDirectory}");
Console.WriteLine($"KernelAssembly: {typeof(Kernel).Assembly.Location}");
Console.WriteLine($"KernelVersion: {typeof(Kernel).Assembly.GetName().Version}");

## 1. Basic Bash Execution

In [ ]:
echo "Hello from Bash!"
echo "Current shell: $BASH_VERSION"

In [ ]:
# Test multiline script
for i in 1 2 3; do
    echo "Iteration $i"
done

## 2. Session Persistence

Environment variables and working directory persist across cells.

In [ ]:
# Set an environment variable
export MY_VAR="Hello from bash"
echo "MY_VAR is now: $MY_VAR"

In [ ]:
# Verify the variable persists
echo "MY_VAR still contains: $MY_VAR"

In [ ]:
# Change directory
cd /tmp
echo "Changed to: $(pwd)"

In [ ]:
# Verify directory persists
echo "Still in: $(pwd)"

## 3. Variable Sharing: Bash → C#

Export a variable in bash, then access it from C#.

In [ ]:
# Create variables to share
export RESULT="42"
export GREETING="Hello from Bash!"
export JSON_DATA='{"name":"test","value":123}'

In [ ]:
#!share --from bash RESULT
#!share --from bash GREETING

Console.WriteLine($"RESULT from bash: {RESULT}");
Console.WriteLine($"GREETING from bash: {GREETING}");

## 4. Variable Sharing: C# → Bash

Define a variable in C#, then use it in bash.

In [ ]:
var apiUrl = "https://api.example.com/data";
var configPath = "/etc/myapp/config.json";

In [ ]:
#!set --name apiUrl --value @csharp:apiUrl
#!set --name configPath --value @csharp:configPath

echo "API URL: $apiUrl"
echo "Config Path: $configPath"

## 5. Error Handling

Non-zero exit codes are reported as failures.

In [ ]:
# This should succeed
echo "This works fine"
true

In [ ]:
# This will fail with exit code 1 (uncomment to test)
# false

## 6. Stderr Output

Standard error is captured separately.

In [ ]:
echo "This goes to stdout"
echo "This goes to stderr" >&2

## 7. Complex Scripts

In [ ]:
# Function definition persists in session
greet() {
    local name=$1
    echo "Hello, $name!"
}

greet "World"

In [ ]:
# Function still available
greet "Polyglot Notebooks"

## Summary

✅ Basic execution works  
✅ Session persistence (export, cd, functions)  
✅ Variable sharing bash → C#  
✅ Variable sharing C# → bash  
✅ Error handling with exit codes  
✅ Stderr capture